In [14]:
from tensorflow.keras.models import load_model

model = load_model(r"D:\FYP\Trial codes\Review 4\output_Isc_newfeatures\final_model_Isc.h5")


In [15]:
model.summary()


Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_2 (InputLayer)        [(None, 256, 256, 3)]        0         []                            
                                                                                                  
 efficientnetb0 (Functional  (None, 8, 8, 1280)           4049571   ['input_2[0][0]']             
 )                                                                                                
                                                                                                  
 global_average_pooling2d (  (None, 1280)                 0         ['efficientnetb0[0][0]']      
 GlobalAveragePooling2D)                                                                          
                                                                                            

In [16]:
import pandas as pd

iv_df = pd.read_excel(
    r"D:\FYP\Datasets\final_industry_dataset\industry_labels_cleaned_final.xlsx"
)


In [17]:
import cv2
import numpy as np
import os

IMG_SIZE = 256

def preprocess_image(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        return None

    # Convert grayscale → 3 channel
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype(np.float32) / 255.0

    return img


In [18]:
from tqdm import tqdm
import numpy as np
import os
import pandas as pd

image_root = r"D:\FYP\Datasets\final_industry_dataset_rgb"
AUX_DIM = 19  # ← FROM MODEL SUMMARY

predictions = []

for _, row in tqdm(iv_df.iterrows(), total=len(iv_df)):
    image_id = row["image_id"]
    voc_true = row["Isc"]

    # --------------------------------------------------
    # Locate image in busbar subfolders
    # --------------------------------------------------
    img_path = None
    for sub in ["3_busbars", "4_busbars", "5_busbars"]:
        candidate = os.path.join(image_root, sub, image_id)
        if os.path.exists(candidate):
            img_path = candidate
            break

    if img_path is None:
        continue

    # --------------------------------------------------
    # Preprocess image (must match training)
    # --------------------------------------------------
    img = preprocess_image(img_path)
    if img is None:
        continue

    # --------------------------------------------------
    # Dummy auxiliary input (neutralized)
    # --------------------------------------------------
    dummy_aux = np.zeros((1, AUX_DIM), dtype=np.float32)

    # --------------------------------------------------
    # Predict
    # --------------------------------------------------
    pred = model.predict(
        [img[None, ...], dummy_aux],
        verbose=0
    )[0][0]

    predictions.append([
        image_id,
        voc_true,
        float(pred)
    ])


100%|██████████| 44562/44562 [1:13:43<00:00, 10.07it/s] 


In [19]:
pred_df = pd.DataFrame(
    predictions,
    columns=["image_id", "isc_true", "isc_pred"]
)

pred_df.to_csv("regression_preds_isc.csv", index=False)

In [21]:
print(pred_df.head())
print(pred_df["isc_pred"].describe())


                               image_id  isc_true  isc_pred
0  10084_35_1_01272020_20_cell_1_2.tiff  8.860558  6.863023
1  10084_35_1_01272020_20_cell_2_3.tiff  8.840771  6.864567
2  10084_35_1_01272020_20_cell_2_4.tiff  8.835577  6.864006
3  10084_35_1_01272020_20_cell_3_1.tiff  8.896506  6.861384
4  10084_35_1_01272020_20_cell_3_2.tiff  8.882539  6.862854
count    44371.000000
mean         6.860783
std          0.005876
min          6.820648
25%          6.858112
50%          6.862124
75%          6.865026
max          6.870499
Name: isc_pred, dtype: float64


In [22]:
pred_df = pd.read_csv("regression_preds_isc.csv")
final_pred_df = (
    pred_df
    .groupby("image_id", as_index=False)
    .agg({
        "isc_true": "first",   # same for all rows of an image
        "isc_pred": "mean"     # average stochastic predictions
    })
)


In [23]:
print(final_pred_df.head())
print(final_pred_df.shape)


                               image_id  isc_true  isc_pred
0  10084_35_1_01272020_20_cell_1_2.tiff  8.860558  6.863023
1  10084_35_1_01272020_20_cell_2_3.tiff  8.840771  6.864567
2  10084_35_1_01272020_20_cell_2_4.tiff  8.835577  6.864006
3  10084_35_1_01272020_20_cell_3_1.tiff  8.896506  6.861384
4  10084_35_1_01272020_20_cell_3_2.tiff  8.882539  6.862854
(44371, 3)


In [24]:
final_pred_df.to_csv("regression_preds_cleaned_isc.csv", index=False)
